# IEMOCAP Dataset — Exploratory Data Analysis

This notebook explores the IEMOCAP 6-class dataset to understand class distributions, VAD score distributions, audio duration statistics, and text length distributions.


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

CSV_PATH = '../data/processed/iemocap_6class.csv'
df = pd.read_csv(CSV_PATH)
print(f'Total utterances: {len(df)}')
df.head()

## 1. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
counts = df['emotion'].value_counts()
colors = ['#e74c3c', '#f1c40f', '#95a5a6', '#3498db', '#e67e22', '#2ecc71']
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title('Overall Emotion Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
for i, (emo, cnt) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, cnt + 10, str(cnt), ha='center', fontsize=10)

# Per-session distribution
session_dist = df.groupby(['session', 'emotion']).size().unstack(fill_value=0)
session_dist.plot(kind='bar', ax=axes[1], stacked=True)
axes[1].set_title('Emotion Distribution per Session', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Session')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../logs/eda_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. VAD Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
dims = ['valence', 'arousal', 'dominance']
titles = ['Valence', 'Arousal', 'Dominance']

for ax, dim, title in zip(axes, dims, titles):
    for emo in df['emotion'].unique():
        subset = df[df['emotion'] == emo][dim]
        ax.hist(subset, bins=20, alpha=0.6, label=emo, density=True)
    ax.set_title(f'{title} Distribution by Emotion', fontsize=12)
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../logs/eda_vad_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Audio Duration Statistics

In [ ]:
df['duration'] = df['end_time'] - df['start_time']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration histogram
axes[0].hist(df['duration'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df['duration'].mean(), color='red', linestyle='--', label=f'Mean: {df["duration"].mean():.2f}s')
axes[0].axvline(8.0, color='orange', linestyle='--', label='Max cutoff: 8s')
axes[0].set_title('Audio Duration Distribution', fontsize=13)
axes[0].set_xlabel('Duration (s)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Duration per emotion (box plot)
df.boxplot(column='duration', by='emotion', ax=axes[1])
axes[1].set_title('Duration by Emotion', fontsize=13)
axes[1].set_xlabel('Emotion')
axes[1].set_ylabel('Duration (s)')
plt.suptitle('')

print(f'Duration stats:\n{df["duration"].describe()}')
print(f'\nUtterances > 8s: {(df["duration"] > 8).sum()} ({100*(df["duration"] > 8).mean():.1f}%)')

plt.tight_layout()
plt.savefig('../logs/eda_duration.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Text Length Distribution

In [ ]:
df['text_len'] = df['text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['text_len'], bins=30, color='coral', edgecolor='white')
ax.axvline(df['text_len'].mean(), color='red', linestyle='--', label=f'Mean: {df["text_len"].mean():.1f} words')
ax.set_title('Transcript Word Count Distribution', fontsize=13)
ax.set_xlabel('Number of Words')
ax.set_ylabel('Count')
ax.legend()
print(f'Text length stats:\n{df["text_len"].describe()}')
plt.tight_layout()
plt.show()

## 5. Sample Spectrograms per Emotion Class

In [ ]:
emotions = ['ang', 'hap', 'neu', 'sad', 'fru', 'exc']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, emo in zip(axes.flat, emotions):
    sample = df[df['emotion'] == emo].sample(1).iloc[0]
    y, sr = librosa.load(sample['audio_path'], sr=16000)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, n_fft=1024, hop_length=256, fmax=8000)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(log_mel, sr=sr, hop_length=256, fmax=8000, x_axis='time', y_axis='mel', ax=ax, cmap='magma')
    ax.set_title(f'{emo.upper()} — "{sample["text"][:40]}"', fontsize=10)
    ax.set_xlabel('')

plt.suptitle('Sample Mel-Spectrograms per Emotion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../logs/eda_spectrograms.png', dpi=150, bbox_inches='tight')
plt.show()